# DATA PREPARATION

In [2]:
# INITIATION: PYTHON AND DISPLAY

# Import libraries
import pandas as pd #
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt #charts
import seaborn as sns #charts

# For better visibility
from IPython.display import display, HTML

# Apply a no-wrap style to prevent text wrapping
display(HTML("""
<style>
.dataframe td { white-space: nowrap; }
</style>
"""))

# Set the option to display all columns
pd.set_option('display.max_columns', None)

# Set the option to display all rows
pd.set_option('display.max_rows', None)

# Set the float format to use a space as a thousands separator and a comma as a decimal separator
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' ').replace('.', ','))

In [3]:
# FILE LOAD

# Read the Excel file into a DataFrame
df_filter = pd.read_excel('2000 Analysis Python CHANGED.xlsx', decimal=',')

# STATISTICAL ANALYSIS

In [11]:
# OLS Regression Model
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Filter the data for the year 2022
data_2022 = df_filter[df_filter['Year'] == 2022]

# Remove rows with missing values in the relevant columns
data_2022_cleaned = data_2022.dropna(subset=['P/F', 'G3','Hint','Drop_Sales','Drop_Employment','Starting_Size','Starting_Profit','Human_Capital', 'Letter', 'LSGrowth_2022_2019'])

# Define the dependent variable
y_2022 = data_2022_cleaned['LSGrowth_2022_2019']

# Define the independent variables
X_2022 = data_2022_cleaned[['P/F', 'G3','Hint','Drop_Sales','Drop_Employment','Starting_Size','Starting_Profit','Human_Capital', 'Letter']]

# Convert categorical variables to dummies
X_2022 = pd.get_dummies(X_2022, drop_first=True)

# Add a constant to the model (intercept)
X_2022 = sm.add_constant(X_2022)

# Convert all boolean columns in the DataFrame to integers
for col in X_2022.columns:
    if X_2022[col].dtype == 'bool':
        X_2022[col] = X_2022[col].map({True: 1, False: 0})

# Fit the OLS model
model_2022 = sm.OLS(y_2022, X_2022).fit()

# Show the summary of the regression
print(model_2022.summary())

# Calculate VIF for each independent variable
vif_data = pd.DataFrame()
vif_data['Variable'] = X_2022.columns
vif_data['VIF'] = [variance_inflation_factor(X_2022.values, i) for i in range(X_2022.shape[1])]

# Show the VIF results
print(vif_data)

                            OLS Regression Results                            
Dep. Variable:     LSGrowth_2022_2019   R-squared:                       0.582
Model:                            OLS   Adj. R-squared:                  0.577
Method:                 Least Squares   F-statistic:                     115.4
Date:                Thu, 10 Oct 2024   Prob (F-statistic):               0.00
Time:                        14:03:40   Log-Likelihood:                -807.83
No. Observations:                1844   AIC:                             1662.
Df Residuals:                    1821   BIC:                             1789.
Df Model:                          22                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   3.1515    